---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI and Data-Driven Marketplaces

### 📋 **Topic**: DataFrames and Marketplace Analysis

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---


# Inside a Marketplace: Airbnb Listings in New York City

A DataFrame is useful because it helps us ask questions about a market. In this notebook, you will use a real snapshot of New York City Airbnb listings to investigate questions such as:

- Where is marketplace supply concentrated?
- What kinds of listings are available?
- How do advertised prices differ across boroughs and room types?
- Which hosts operate more than one listing?
- What can the data tell us—and what can it **not** tell us?

The goal is not to memorize pandas commands. The goal is to learn what you can do with marketplace data, how to ask a coding agent for help, and how to check whether the result makes sense.


## How to work with a coding agent

You may use a coding AI agent throughout this notebook. A productive workflow is:

1. **State the marketplace question** in plain English.
2. **Name the DataFrame and columns** the agent should use.
3. Ask for **short pandas code plus a plain-English explanation**.
4. Run the code yourself.
5. Check the number of rows, the column names, and a few examples.
6. Explain the result in your own words.

> Example prompt: “Using the DataFrame `listings`, show five private-room listings in the Bronx with an advertised nightly price of at most $150. Show only the neighborhood, room type, price, and recent reviews. Sort from lowest to highest price. Explain the code line by line.”

If code fails, give the agent the **full error message** and say what you expected to happen. Do not ask it to guess from a cropped screenshot.


## 1. The marketplace and the data

Airbnb connects hosts who offer places to stay with guests who want accommodations. The platform helps participants discover one another, communicate, transact, and build trust.

This notebook uses the **New York City summary listings snapshot from August 10, 2026**, published by [Inside Airbnb](https://insideairbnb.com/get-the-data/). Inside Airbnb publishes a [data dictionary](https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit?usp=sharing), documents its assumptions, and licenses the data under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

The exact snapshot is saved in this course repository so that everyone analyzes the same data. Inside Airbnb is not associated with or endorsed by Airbnb. The data come from public information and represent a snapshot—not Airbnb's private transaction records.


### Historical context: Why does Inside Airbnb exist?

Inside Airbnb is not a neutral corporate data portal. It is a **mission-driven data and advocacy project** founded by artist, activist, and technologist Murray Cox in February 2015. Its roots go back to 2014, when Cox helped young people in Bedford-Stuyvesant use maps, data, and visualization to study gentrification. That experience helped inspire a bigger question: what could public Airbnb listings reveal about short-term rentals and residential neighborhoods? Read [Inside Airbnb's account of its history](https://insideairbnb.com/about/).

The project became influential because outsiders could not see Airbnb's private marketplace data. Inside Airbnb reconstructed part of the market from information visible on Airbnb's public website, then used the results to ask questions about entire-home rentals, hosts with multiple listings, housing, and regulation. Journalists, researchers, advocates, and public officials could analyze a market that had previously been difficult to observe. A [2017 WIRED profile](https://www.wired.com/2017/02/a-lone-data-whiz-is-fighting-airbnb-and-winning/) describes how the project affected public debate and pushed Airbnb toward greater transparency.

The relationship was contentious. In 2016, Cox and author Tom Slee [argued that Airbnb had removed more than 1,000 New York City entire-home listings shortly before taking a snapshot for a public data release](https://insideairbnb.com/reports/how-airbnbs-data-hid-the-facts-in-new-york-city.pdf), making the marketplace look less dominated by commercial operators. That is **their analysis and allegation**, not a neutral fact we should accept without scrutiny. Airbnb responded that scraped data can be flawed and can present an inaccurate picture of its community, while saying that it was releasing more information about hosts, guests, and removed listings.

> **Did Airbnb try to shut Inside Airbnb down?** You may hear the story described that way, but I could not verify a direct lawsuit or cease-and-desist against the project from reliable sources. The well-documented story is a struggle over **data access, measurement, public relations, and regulation**—a platform and its critics offering competing accounts of how the marketplace worked.

**Marketplace lesson:** Data access is power. A platform sees transactions and behavior that outsiders cannot see. Independent datasets can improve accountability, but they also reflect the collector's purpose, methods, and assumptions. Inside Airbnb openly identifies itself as an advocacy project, and its [data assumptions](https://insideairbnb.com/data-assumptions/) explain important limitations. As you work, keep asking: **Who collected these data? Why? What can we observe—and what remains invisible?**


### 1.1 Start with a DataFrame we can see all at once

Before opening thirty thousand rows, we will create five fictional marketplace listings. A dictionary supplies the columns, and each list supplies the values in that column. This makes the row-and-column structure easy to inspect.


In [ ]:
import pandas as pd

sample_listings = pd.DataFrame(
    {
        "borough": ["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"],
        "room_type": ["Private room", "Entire home/apt", "Hotel room", "Private room", "Entire home/apt"],
        "nightly_price": [85, 190, 420, 110, 140],
        "reviews_last_year": [12, 5, 22, 0, 8],
    }
)

sample_listings

Every list contains five values, so pandas creates five rows. The dictionary keys become column names. Real datasets work the same way; they simply contain more rows and columns.


### 1.2 Load the real data

The code below uses pandas to read the CSV file, checks that the expected columns are present, and gives the columns friendlier names. You do not need to memorize this setup code. Ask your coding agent to explain any line that is unclear.


In [ ]:
from pathlib import Path

data_file = Path("../data/inside-airbnb-nyc-listings-2026-08-10.csv")
raw_listings = pd.read_csv(data_file)

column_names = {
    "id": "listing_id",
    "host_id": "host_id",
    "neighbourhood_group": "borough",
    "neighbourhood": "neighborhood",
    "room_type": "room_type",
    "price": "nightly_price",
    "minimum_nights": "minimum_nights",
    "number_of_reviews": "total_reviews",
    "reviews_per_month": "reviews_per_month",
    "calculated_host_listings_count": "host_listing_count",
    "availability_365": "available_days",
    "number_of_reviews_ltm": "reviews_last_year",
}

missing_columns = set(column_names) - set(raw_listings.columns)
assert not missing_columns, f"Missing expected columns: {missing_columns}"

listings = raw_listings.rename(columns=column_names)[list(column_names.values())].copy()

print("Dataset loaded successfully.")

### 1.3 What does one row represent?

Each row represents one listing observed in the New York City marketplace snapshot. Each column records an attribute of that listing, its host, its advertised terms, or its review history.


In [ ]:
listings.head()

### 1.4 A small data dictionary

| Column | Meaning |
|---|---|
| `listing_id` | Unique listing identifier |
| `host_id` | Unique host identifier |
| `borough` | NYC borough |
| `neighborhood` | Neighborhood assigned to the listing |
| `room_type` | Entire place, private room, shared room, or hotel room |
| `nightly_price` | Advertised daily price in local currency |
| `minimum_nights` | Minimum stay shown for the listing |
| `total_reviews` | Reviews accumulated by the listing |
| `reviews_last_year` | Reviews during the last 12 months |
| `host_listing_count` | Listings operated by the host in this NYC snapshot |
| `available_days` | Days marked available during the next 365 days |

Availability does **not** tell us occupancy: a date can be unavailable because it was booked or because the host blocked it. Reviews are also not the same as bookings or revenue.


## 2. Meet the DataFrame

Before answering a marketplace question, check the size, columns, and data types. This helps you—and your coding agent—understand what is actually available.


In [ ]:
print(f"Rows: {listings.shape[0]:,}")
print(f"Columns: {listings.shape[1]}")
print("Column names:")
print(listings.columns.tolist())

In [ ]:
listings.dtypes

### Two quick summaries

The `.info()` method summarizes row counts, missing values, and data types. The `.describe()` method summarizes numerical columns. These are useful first requests when you or your coding agent encounter an unfamiliar DataFrame.


In [ ]:
listings.info()

In [ ]:
listings[[
    "nightly_price",
    "minimum_nights",
    "total_reviews",
    "reviews_last_year",
    "host_listing_count",
]].describe().round(1)

### Check missing information

Real marketplace data are rarely complete. Missing values are information we must notice before computing an answer.


In [ ]:
missing_summary = (
    listings.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_rows")
)

missing_summary

Many listings are missing an advertised price in this snapshot. For price comparisons, we will keep listings with a recorded price between $20 and $1,000. This focuses the classroom analysis and prevents extreme values from dominating summaries; it does **not** prove that values outside this range are errors.


In [ ]:
analysis_listings = (
    listings
    .dropna(subset=["nightly_price"])
    .query("20 <= nightly_price <= 1000")
    .copy()
)

print(f"All listings: {len(listings):,}")
print(f"Listings used for price analysis: {len(analysis_listings):,}")

## 3. Marketplace question: Where is supply concentrated?

A marketplace operator needs to understand where supply exists. The `value_counts()` method counts how many rows belong to each category.


In [ ]:
borough_supply = (
    listings["borough"]
    .value_counts()
    .rename_axis("borough")
    .reset_index(name="listings")
)

borough_supply

In [ ]:
room_type_supply = (
    listings["room_type"]
    .value_counts()
    .rename_axis("room_type")
    .reset_index(name="listings")
)

room_type_supply

**What to notice:** Manhattan and Brooklyn contain most listings in this snapshot. Entire homes/apartments and private rooms dominate the observed supply. Counts describe the snapshot; they do not tell us how many stays actually occurred.


## 4. Marketplace question: What could a budget traveler find?

Filtering keeps only rows that satisfy conditions. Selecting columns keeps the result focused on the decision. Sorting puts the most relevant options first.


In [ ]:
bronx_private_rooms = (
    analysis_listings[
        (analysis_listings["borough"] == "Bronx")
        & (analysis_listings["room_type"] == "Private room")
        & (analysis_listings["nightly_price"] <= 150)
    ]
    [["neighborhood", "room_type", "nightly_price", "reviews_last_year"]]
    .sort_values("nightly_price")
)

bronx_private_rooms.head(10)

### Select more than one category with `.isin()`

Instead of writing several `or` conditions, `.isin()` checks whether each row belongs to a list of acceptable categories. Here we look at private rooms in Brooklyn or Queens.


In [ ]:
brooklyn_or_queens = (
    analysis_listings[
        analysis_listings["borough"].isin(["Brooklyn", "Queens"])
        & (analysis_listings["room_type"] == "Private room")
    ]
    [["borough", "neighborhood", "nightly_price", "reviews_last_year"]]
    .sort_values(["borough", "nightly_price"])
)

print(f"Matching listings: {len(brooklyn_or_queens):,}")
brooklyn_or_queens.head(10)

### Try it with your coding agent

Choose a borough, room type, and budget that interest you. Ask your coding agent to modify the previous analysis. Require it to use `analysis_listings`, return no more than five columns, sort the result, and explain every condition.

After running the code, verify:

- Are all rows from the borough you requested?
- Are all prices within your budget?
- Does the sort order match your request?


In [ ]:
# Paste your agent-assisted filtering code below this comment.


## 5. Marketplace question: Which listings receive attention?

Sorting can reveal listings with many recent reviews. Reviews can be a useful signal of activity, but they are not a complete measure of bookings, quality, or revenue.


In [ ]:
most_reviewed_recently = (
    analysis_listings
    .sort_values("reviews_last_year", ascending=False)
    [[
        "borough",
        "neighborhood",
        "room_type",
        "nightly_price",
        "reviews_last_year",
    ]]
    .head(10)
)

most_reviewed_recently

## 6. Derive information the marketplace did not provide directly

A DataFrame becomes more useful when we derive new columns from existing ones. Here we create two simple indicators:

- whether a listing received at least one review during the last year; and
- whether its host operates more than one listing in this NYC snapshot.


In [ ]:
analysis_listings["has_recent_review"] = analysis_listings["reviews_last_year"] > 0
analysis_listings["multi_listing_host"] = analysis_listings["host_listing_count"] > 1

analysis_listings[[
    "listing_id",
    "reviews_last_year",
    "has_recent_review",
    "host_listing_count",
    "multi_listing_host",
]].head()

### Derive a numerical column with arithmetic

We can multiply the advertised nightly price by the minimum number of nights to create a simplified `advertised_minimum_stay_cost`. This excludes taxes, fees, discounts, and price changes, so it is an illustration—not the amount a guest necessarily paid.


In [ ]:
analysis_listings["advertised_minimum_stay_cost"] = (
    analysis_listings["nightly_price"] * analysis_listings["minimum_nights"]
)

analysis_listings[[
    "nightly_price",
    "minimum_nights",
    "advertised_minimum_stay_cost",
]].head()

In [ ]:
recent_review_share = analysis_listings["has_recent_review"].mean() * 100
multi_listing_share = analysis_listings["multi_listing_host"].mean() * 100

print(f"Listings with a review in the last year: {recent_review_share:.1f}%")
print(f"Listings operated by multi-listing hosts: {multi_listing_share:.1f}%")

The label `multi_listing_host` is descriptive. It does not prove that a host is a professional business, and it does not tell us why the host operates multiple listings.


## 7. Compare marketplace segments with `groupby()`

A marketplace is rarely one uniform market. `groupby()` lets us compare segments such as boroughs and room types. We will use the median price because extreme advertised prices can strongly affect the mean.


In [ ]:
borough_summary = (
    analysis_listings
    .groupby("borough", as_index=False)
    .agg(
        listings=("listing_id", "count"),
        median_price=("nightly_price", "median"),
        median_minimum_nights=("minimum_nights", "median"),
        recent_review_share=("has_recent_review", "mean"),
    )
)

borough_summary["recent_review_share"] = (
    borough_summary["recent_review_share"] * 100
).round(1)
borough_summary["median_price"] = borough_summary["median_price"].round(0)

borough_summary.sort_values("median_price", ascending=False)

**What to notice:** Manhattan has the highest median advertised price in this focused sample. That does not mean “being in Manhattan” causes a particular price: room type, neighborhood, host choices, and many unobserved differences also matter.


In [ ]:
borough_room_summary = (
    analysis_listings
    .groupby(["borough", "room_type"], as_index=False)
    .agg(
        listings=("listing_id", "count"),
        median_price=("nightly_price", "median"),
    )
    .query("listings >= 50")
    .sort_values("median_price", ascending=False)
)

borough_room_summary

This comparison is more useful than comparing borough averages alone because an entire home and a private room are different products. Marketplace analysis improves when we compare like with like.


## 8. Method chaining: Turn one question into one pipeline

Suppose a marketplace team asks:

> Among private rooms priced below $300, which boroughs have the most listings, and what is the median advertised price?

The following pipeline answers the question from top to bottom. Read each line as one instruction.


In [ ]:
private_room_market = (
    analysis_listings
    .query("room_type == 'Private room' and nightly_price < 300")
    .groupby("borough", as_index=False)
    .agg(
        listings=("listing_id", "count"),
        median_price=("nightly_price", "median"),
        neighborhoods=("neighborhood", "nunique"),
    )
    .sort_values("listings", ascending=False)
)

private_room_market

### Ask your coding agent to translate—not merely generate

Use this prompt:

> “Translate the `private_room_market` pipeline into numbered plain-English steps. For each pandas method, state what rows or columns exist immediately after that line. Then identify one check I can run to confirm the result.”

Being able to explain generated code is more valuable than memorizing its punctuation.


## 9. Your AI-assisted marketplace investigation

Choose **one** question below and ask your coding agent to help you answer it with `analysis_listings`:

1. **Traveler question:** For a borough and budget of your choice, which room type provides the most options?
2. **Host question:** Which neighborhoods with at least 100 observed listings have the highest median advertised prices?
3. **Marketplace question:** How do single-listing and multi-listing hosts differ in advertised price and recent-review activity?
4. **Your question:** Ask something else that the available columns can actually answer.

Ask the agent for:

- one short pandas pipeline;
- a result with no more than six columns and twenty rows;
- one validation check; and
- a warning about what cannot be concluded.


In [ ]:
# Paste and run your agent-assisted marketplace investigation below.


### Save a result as a CSV file

A DataFrame can be exported for another person or program. The example below saves the private-room market summary in the course `temp` folder. Replace `private_room_market` with the name of your own result when you are ready to export it.


In [ ]:
result_to_save = private_room_market
output_file = Path("../temp/airbnb-marketplace-investigation.csv")
output_file.parent.mkdir(exist_ok=True)

result_to_save.to_csv(output_file, index=False)
print(f"Saved {len(result_to_save)} rows to {output_file}")

### Explain your result

Add a Markdown cell below this one and answer:

1. What marketplace question did you ask?
2. What did the result show?
3. How did you check the code or output?
4. What can you **not** conclude from this snapshot?


## 10. Takeaways

You can now use a DataFrame to:

- create a small DataFrame from a dictionary of lists;
- inspect the size, columns, types, and missing values of marketplace data;
- select, filter, and sort listings;
- derive useful indicators from existing columns;
- compare marketplace segments with `groupby()`;
- combine operations in a readable pipeline;
- export a result as a CSV file; and
- collaborate with a coding agent while checking its work.

Most importantly, you practiced separating **what the data show** from **why the pattern exists**. A DataFrame can reveal patterns and help generate questions. It does not automatically establish causality.
